In [1]:
import json
import os

from doc_chat.rag.multi_tenant_vector_store import MultiTenantVectorStore

In [2]:
# set the number of files to process, None means all files
number_of_articles = 3
file_path = '../../data/squad/raw/train-v1.1.json'

In [3]:
chroma_persist_directory = '/Users/patrick/projects/doc-chat/app_data/chroma_eval'
if not os.path.exists(chroma_persist_directory):
    os.makedirs(chroma_persist_directory)
user_id = 'squad_semantic_search_text_embedding_3_small'
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory,
                                      embedding_model='text-embedding-3-small')

Using Chroma persist directory: /Users/patrick/projects/doc-chat/app_data/chroma_eval


In [6]:
# index the articles
class Split:
    def __init__(self, content, page=0):
        self.page_content = content
        self.metadata = {"page": page}


with open(file_path, 'r') as f:
    squad_data = json.load(f)

print('Number of articles:', len(squad_data['data']))
articles = squad_data['data']

for article_idx, article in enumerate(articles):
    if number_of_articles and article_idx >= number_of_articles:
        break
    title = article['title']
    paragraphs = article['paragraphs']
    document_splits = []
    for p_idx, paragraph in enumerate(paragraphs):
        context = paragraph['context']
        document_splits.append(Split(content=context, page=p_idx))

    print(f"Indexing {len(document_splits)} chunks for article '{title}'")
    collection_id = f"article_{article_idx}"
    vector_store.delete_collection(user_id=user_id, collection_id=collection_id)
    vector_store.create_document_collection(user_id=user_id,
                                            collection_id=collection_id,
                                            document_splits=document_splits,
                                            file_name=title)

collections = vector_store.get_collections(user_id=user_id)
print(collections)

Number of articles: 442
Indexing 55 chunks for article 'University_of_Notre_Dame'

Indexing 66 chunks for article 'Beyoncé'

Indexing 44 chunks for article 'Montana'

['article_0', 'article_1', 'article_2']
